In [4]:
import torch
if torch.cuda.is_available():
    device='cuda'
else:
    device='cpu'
device

'cuda'

In [ ]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_set = torchvision.datasets.CIFAR10(
    root='datasets', train=True, download=True, transform=toTensor
)
test_set = torchvision.datasets.CIFAR10(
    root='datasets', train=False, download=True, transform=toTensor
)

100%|██████████| 170M/170M [00:03<00:00, 42.9MB/s] 


In [3]:
torch.manual_seed(42)
train_set, valid_set = torch.utils.data.random_split(
    train_and_valid_set,[45_000, 5_000]
)

In [26]:
from torch.utils.data import DataLoader
batch_size=128
train_loader = DataLoader(train_set, batch_size=batch_size)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

In [6]:
import torch.nn as nn
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

In [7]:
def build_deep_model(n_hidden, n_neurons, n_inputs, n_outputs):
    layers = [nn.Flatten(), nn.Linear(n_inputs, n_neurons), nn.SiLU()]
    for _ in range(n_hidden - 1):
        layers += [nn.Linear(n_neurons, n_neurons),nn.SiLU()]
    layers += [nn.Linear(n_neurons, n_outputs)]
    model = torch.nn.Sequential(*layers)
    model.apply(use_he_init)
    return model

In [8]:
torch.manual_seed(42)
model = build_deep_model(
    n_hidden=20, n_neurons=100, n_inputs=3 *32 *32, n_outputs=10
).to(device)

In [10]:
%pip install torchmetrics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 30.8 MB/s eta 0:00:00


In [13]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
        return metric.compute()

In [14]:
import time
import torchmetrics
def train_with_early_stopping(model, optimizer, loss_fn, metric, train_loader,
                              valid_loader, n_epochs, patience=10,
                              checkpoint_path=None, scheduler=None):
    checkpoint_path= checkpoint_path or 'my_checkpoint.pt'
    history = {'train_losses':[],'train_metrics':[],'valid_metrics':[]}
    best_metric = 0.0
    patience_counter = 0
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        t0 = time.time()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)

        train_metric = metric.compute().item()
        valid_metric = evaluate_tm(model, valid_loader, metric).item()
        if valid_metric > best_metric:
            torch.save(model.state_dict(), checkpoint_path)
            best_metric = valid_metric
            best = " (best)"
            patience_counter = 0
        else:
            patience_counter += 1
            best = ""
        
        t1 = time.time()
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}{best}"
              f" in {t1 - t0:.1f}s"
        )
        if scheduler is not None:
            scheduler.step()
        if patience_counter >= patience:
            print('Early stopping!')
            break
    model.load_state_dict(torch.load(checkpoint_path))
    return history

In [15]:
optimizer = torch.optim.NAdam(model.parameters(), lr=2e-3)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to(device)

In [16]:
n_epochs=100
history = train_with_early_stopping(model, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 2.1743, train metric: 0.1644, valid metric: 0.1786 (best) in 14.2s
Epoch 2/100, train loss: 2.0251, train metric: 0.2202, valid metric: 0.2406 (best) in 12.7s
Epoch 3/100, train loss: 1.9205, train metric: 0.2679, valid metric: 0.2634 (best) in 13.1s
Epoch 4/100, train loss: 2.0932, train metric: 0.2575, valid metric: 0.2656 (best) in 13.1s
Epoch 5/100, train loss: 1.8331, train metric: 0.3129, valid metric: 0.3068 (best) in 13.2s
Epoch 6/100, train loss: 1.7872, train metric: 0.3343, valid metric: 0.3168 (best) in 12.9s
Epoch 7/100, train loss: 1.7739, train metric: 0.3452, valid metric: 0.3374 (best) in 12.9s
Epoch 8/100, train loss: 1.7493, train metric: 0.3562, valid metric: 0.3332 in 12.9s
Epoch 9/100, train loss: 1.7258, train metric: 0.3657, valid metric: 0.3116 in 12.6s
Epoch 10/100, train loss: 1.7291, train metric: 0.3675, valid metric: 0.2790 in 13.1s
Epoch 11/100, train loss: 1.7267, train metric: 0.3672, valid metric: 0.3638 (best) in 12.6s
Epoch 1

In [17]:
def build_deep_model_with_batch_norm(n_hidden, n_neurons, n_inputs, n_outputs):
    layers = [nn.Flatten(), nn.Linear(n_inputs, n_neurons),
              nn.BatchNorm1d(n_neurons), nn.SiLU()]
    for _ in range(n_hidden - 1):
        layers += [nn.Linear(n_neurons, n_neurons),
                   nn.BatchNorm1d(n_neurons), nn.SiLU()]
    layers += [nn.Linear(n_neurons, n_outputs)]
    model = torch.nn.Sequential(*layers)
    model.apply(use_he_init)
    return model

In [18]:
torch.manual_seed(42)
model = build_deep_model_with_batch_norm(
    n_hidden=20, n_neurons=100, n_inputs= 3*32*32, n_outputs=10
).to(device)

In [19]:
optimizer = torch.optim.NAdam(model.parameters(), lr=2e-3)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to(device)

In [20]:
n_epochs=100
history = train_with_early_stopping(model, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 1.9239, train metric: 0.2956, valid metric: 0.3274 (best) in 14.8s
Epoch 2/100, train loss: 1.6310, train metric: 0.4108, valid metric: 0.3422 (best) in 14.4s
Epoch 3/100, train loss: 1.5102, train metric: 0.4570, valid metric: 0.3444 (best) in 14.5s
Epoch 4/100, train loss: 1.4245, train metric: 0.4909, valid metric: 0.3702 (best) in 15.2s
Epoch 5/100, train loss: 1.3555, train metric: 0.5191, valid metric: 0.3960 (best) in 15.0s
Epoch 6/100, train loss: 1.2951, train metric: 0.5411, valid metric: 0.4008 (best) in 14.5s
Epoch 7/100, train loss: 1.2437, train metric: 0.5605, valid metric: 0.3852 in 14.5s
Epoch 8/100, train loss: 1.1990, train metric: 0.5760, valid metric: 0.3650 in 15.0s
Epoch 9/100, train loss: 1.1582, train metric: 0.5908, valid metric: 0.3780 in 14.4s
Epoch 10/100, train loss: 1.1207, train metric: 0.6042, valid metric: 0.3642 in 14.1s
Epoch 11/100, train loss: 1.0853, train metric: 0.6207, valid metric: 0.3744 in 14.4s
Epoch 12/100, train l

In [24]:
class Standardize(nn.Module):
    def __init__(self, sample):
        super().__init__()
        flat = torch.flatten(sample, start_dim=1)
        mean = flat.mean(dim=0, keepdim=True)
        std= flat.std(dim=0, keepdim=True)
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
        
    def forward(self, x):
        return (x - self.mean) / self.std

In [27]:
all_images = torch.stack([img for img, _ in train_set])
standardize = Standardize(all_images)

In [28]:
def use_lecun_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, mode='fan_in',
                                nonlinearity='linear')
        nn.init.zeros_(module.bias)

In [29]:
def build_deep_model_with_selu(n_hidden, n_neurons, n_inputs, n_outputs):
    layers = [nn.Flatten(), standardize,
              nn.Linear(n_inputs, n_neurons), nn.SELU()]
    for _ in range(n_hidden - 1):
        layers += [nn.Linear(n_neurons, n_neurons), nn.SELU()]

    layers += [nn.Linear(n_neurons, n_outputs)]
    model = torch.nn.Sequential(*layers)
    model.apply(use_lecun_init)
    return model

In [30]:
torch.manual_seed(42)
model = build_deep_model_with_selu(
    n_hidden=20, n_neurons=100, n_inputs=3 * 32 * 32, n_outputs=10
).to(device)

In [31]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

In [32]:
n_epochs = 100
history = train_with_early_stopping(model, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 2.0557, train metric: 0.2615, valid metric: 0.3162 (best) in 12.8s
Epoch 2/100, train loss: 1.8506, train metric: 0.3348, valid metric: 0.3490 (best) in 12.6s
Epoch 3/100, train loss: 1.7769, train metric: 0.3622, valid metric: 0.3638 (best) in 13.0s
Epoch 4/100, train loss: 1.7250, train metric: 0.3836, valid metric: 0.3754 (best) in 12.9s
Epoch 5/100, train loss: 1.6809, train metric: 0.4014, valid metric: 0.3854 (best) in 12.6s
Epoch 6/100, train loss: 1.6425, train metric: 0.4159, valid metric: 0.3906 (best) in 12.6s
Epoch 7/100, train loss: 1.6091, train metric: 0.4268, valid metric: 0.3982 (best) in 12.6s
Epoch 8/100, train loss: 1.5797, train metric: 0.4387, valid metric: 0.4058 (best) in 12.9s
Epoch 9/100, train loss: 1.5531, train metric: 0.4487, valid metric: 0.4134 (best) in 12.6s
Epoch 10/100, train loss: 1.5293, train metric: 0.4569, valid metric: 0.4188 (best) in 12.6s
Epoch 11/100, train loss: 1.5073, train metric: 0.4654, valid metric: 0.4240 (b

In [33]:
def build_deep_model_with_alpha_dropout(n_hidden, n_neurons, n_inputs,
                                        n_outputs, dropout_rate):
    layers = [nn.Flatten(), standardize,
              nn.Linear(n_inputs, n_neurons), nn.SELU(),
              nn.AlphaDropout(dropout_rate)]
    for _ in range(n_hidden - 1):
        layers += [nn.Linear(n_neurons, n_neurons), nn.SELU(),
                   nn.AlphaDropout(dropout_rate)]

    layers += [nn.Linear(n_neurons, n_outputs)]
    model = torch.nn.Sequential(*layers)
    model.apply(use_lecun_init)
    return model

In [34]:
torch.manual_seed(42)
model = build_deep_model_with_alpha_dropout(
    n_hidden=20, n_neurons=100, n_inputs=3 * 32 * 32, n_outputs=10,
    dropout_rate=0.1
).to(device)

In [35]:
optimizer = torch.optim.NAdam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

In [36]:
n_epochs = 100
history = train_with_early_stopping(model, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 2.1558, train metric: 0.1955, valid metric: 0.1844 (best) in 14.0s
Epoch 2/100, train loss: 1.9809, train metric: 0.2380, valid metric: 0.2258 (best) in 14.0s
Epoch 3/100, train loss: 1.9386, train metric: 0.2576, valid metric: 0.2634 (best) in 14.6s
Epoch 4/100, train loss: 1.8807, train metric: 0.2849, valid metric: 0.3072 (best) in 13.9s
Epoch 5/100, train loss: 1.8526, train metric: 0.3002, valid metric: 0.2974 in 13.6s
Epoch 6/100, train loss: 1.8052, train metric: 0.3280, valid metric: 0.3578 (best) in 13.8s
Epoch 7/100, train loss: 1.7608, train metric: 0.3478, valid metric: 0.3650 (best) in 13.6s
Epoch 8/100, train loss: 1.7331, train metric: 0.3635, valid metric: 0.3748 (best) in 13.9s
Epoch 9/100, train loss: 1.7055, train metric: 0.3746, valid metric: 0.3750 (best) in 13.7s
Epoch 10/100, train loss: 1.6938, train metric: 0.3785, valid metric: 0.3918 (best) in 13.8s
Epoch 11/100, train loss: 1.6580, train metric: 0.3943, valid metric: 0.3928 (best) in

In [37]:
torch.manual_seed(42)
model = build_deep_model_with_alpha_dropout(
    n_hidden=20, n_neurons=100, n_inputs=3 * 32 * 32, n_outputs=10,
    dropout_rate=0.1
).to(device)

In [38]:
n_epochs = 60
optimizer = torch.optim.NAdam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, epochs=n_epochs, steps_per_epoch=len(train_loader), max_lr=1e-2)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

In [39]:
history = train_with_early_stopping(model, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs,
                                    patience=20, scheduler=scheduler)

Epoch 1/60, train loss: 2.2694, train metric: 0.1694, valid metric: 0.2430 (best) in 14.1s
Epoch 2/60, train loss: 2.0259, train metric: 0.2210, valid metric: 0.2292 in 13.8s
Epoch 3/60, train loss: 1.9599, train metric: 0.2409, valid metric: 0.2670 (best) in 13.8s
Epoch 4/60, train loss: 1.9165, train metric: 0.2584, valid metric: 0.2676 (best) in 13.8s
Epoch 5/60, train loss: 1.8840, train metric: 0.2759, valid metric: 0.2986 (best) in 13.8s
Epoch 6/60, train loss: 1.8596, train metric: 0.2943, valid metric: 0.3142 (best) in 14.8s
Epoch 7/60, train loss: 1.8065, train metric: 0.3216, valid metric: 0.3386 (best) in 14.0s
Epoch 8/60, train loss: 1.7636, train metric: 0.3386, valid metric: 0.3660 (best) in 13.7s
Epoch 9/60, train loss: 1.7287, train metric: 0.3585, valid metric: 0.3542 in 13.9s
Epoch 10/60, train loss: 1.7030, train metric: 0.3712, valid metric: 0.3778 (best) in 13.9s
Epoch 11/60, train loss: 1.6750, train metric: 0.3864, valid metric: 0.3848 (best) in 14.2s
Epoch 12/60